# Estudo de caso: teste de permutação com ChickData

Este notebook ilustra a aplicação de um teste de permutação a dados de peso de frangos alimentados com dois tipos de ração: `meatmeal` e `casein`. O objetivo é construir a distribuição nula diretamente por redistribuição dos rótulos de grupo e interpretar o valor-p no contexto do desenho e da hipótese adotados.


## 1. Pergunta de pesquisa e hipóteses

A variável resposta é `weight`, o peso do frango, e a variável de grupo é `feed`, o tipo de ração. Usaremos como estatística a diferença entre médias:

$$T = \bar X_{\text{casein}} - \bar X_{\text{meatmeal}}.$$

O teste de permutação irrestrita pressupõe que, sob a hipótese nula relevante, os rótulos das rações sejam intercambiáveis entre os frangos. Neste estudo, isso corresponde à hipótese de que os dois grupos têm a mesma distribuição de pesos:

$$H_0: F_{\text{casein}} = F_{\text{meatmeal}}$$

contra a alternativa bilateral

$$H_a: F_{\text{casein}} \ne F_{\text{meatmeal}}.$$

A escolha da diferença de médias torna o teste particularmente sensível a diferenças de localização média. Não se deve interpretar uma permutação irrestrita automaticamente como teste exato apenas de igualdade de médias quando os grupos podem diferir em dispersão ou forma.


In [ ]:
from math import comb
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import ttest_ind

ALPHA = 0.05
B = 10_000
rng = np.random.default_rng(1625)

sns.set_theme(style='whitegrid', palette='Set2')


In [ ]:
arquivo = Path('../data/ChickData.csv')
dados = pd.read_csv(arquivo)

dados['feed'] = pd.Categorical(
    dados['feed'],
    categories=['meatmeal', 'casein'],
    ordered=True,
)
dados.head()


## 2. Descrição dos grupos

A análise exploratória informa a escala dos dados, a diferença observada e eventuais assimetrias ou valores extremos. Ela não substitui o teste: diferenças amostrais podem ocorrer mesmo sob a hipótese nula.


In [ ]:
resumo = dados.groupby('feed', observed=False)['weight'].agg(
    n='count',
    media='mean',
    desvio_padrao='std',
    mediana='median',
    minimo='min',
    maximo='max',
)
resumo.round(2)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(
    data=dados, x='feed', y='weight', hue='feed',
    palette='Set2', legend=False, ax=axes[0],
)
sns.stripplot(
    data=dados, x='feed', y='weight', color='black',
    jitter=0.10, alpha=0.7, ax=axes[0],
)
axes[0].set(title='Pesos observados por ração', xlabel='Ração', ylabel='Peso')

for feed, cor in zip(dados['feed'].cat.categories, sns.color_palette('Set2', 2)):
    axes[1].hist(
        dados.loc[dados['feed'] == feed, 'weight'],
        bins=5, alpha=0.55, label=feed, color=cor,
    )
axes[1].set(title='Distribuições empíricas dos pesos', xlabel='Peso', ylabel='Frequência')
axes[1].legend(title='Ração')

plt.tight_layout()


**Leitura inicial.** Compare as médias, medianas e dispersões dos dois grupos. O deslocamento horizontal dos pontos sobrepostos no boxplot é apenas *jitter*: ele evita sobreposição visual e não representa outra variável. Com amostras pequenas, os histogramas são apenas uma indicação da forma dos dados; eles não permitem estabelecer a distribuição populacional.


## 3. Estatística observada e atribuições possíveis

Os pesos permanecem fixos durante a permutação. Em cada redistribuição, alteramos somente os rótulos, preservando os tamanhos originais dos grupos. Com $$n_1$$ frangos em um grupo e $$n_2$$ no outro, o número de atribuições distintas é:

$$\binom{n_1+n_2}{n_1}.$$

Como esse número é relativamente grande neste caso, estimaremos a distribuição nula com $$B=10\,000$$ atribuições aleatórias, em vez de enumerar todas.


In [ ]:
meatmeal = dados.loc[dados['feed'] == 'meatmeal', 'weight'].to_numpy()
casein = dados.loc[dados['feed'] == 'casein', 'weight'].to_numpy()

estatistica_observada = casein.mean() - meatmeal.mean()
n_meatmeal = len(meatmeal)
n_casein = len(casein)
num_atribuicoes = comb(n_meatmeal + n_casein, n_casein)

print(f'n_meatmeal = {n_meatmeal}')
print(f'n_casein = {n_casein}')
print(f'Diferença observada (casein - meatmeal) = {estatistica_observada:.2f}')
print(f'Atribuições possíveis preservando os tamanhos: {num_atribuicoes:,}')


## 4. Construção da distribuição nula por permutação

Sob a hipótese nula e o esquema de permutação adotado, qualquer atribuição de rótulos que preserve os tamanhos dos grupos é igualmente compatível com os dados. A célula abaixo sorteia uma dessas atribuições em cada repetição e recalcula a diferença de médias.

Como a alternativa é bilateral, consideraremos tão extremas quanto a observada as estatísticas cuja magnitude satisfaça:

$$|T_b| \geq |T_{\text{obs}}|.$$

Para Monte Carlo, usamos a correção:

$$\widehat p = \frac{K+1}{B+1},$$

em que $$K$$ é a quantidade de permutações extremas.


In [ ]:
valores_combinados = np.concatenate([meatmeal, casein])
estatisticas_permutadas = np.empty(B)

for b in range(B):
    valores_permutados = rng.permutation(valores_combinados)
    grupo_casein = valores_permutados[:n_casein]
    grupo_meatmeal = valores_permutados[n_casein:]
    estatisticas_permutadas[b] = grupo_casein.mean() - grupo_meatmeal.mean()

k_extremas = np.count_nonzero(
    np.abs(estatisticas_permutadas) >= abs(estatistica_observada)
)
p_permutacao = (k_extremas + 1) / (B + 1)

print(f'Permutações extremas: {k_extremas}')
print(f'p-valor por permutação: {p_permutacao:.4f}')


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

ax.hist(
    estatisticas_permutadas, bins=35, density=True,
    color='#4C78A8', edgecolor='white', alpha=0.85,
)
ax.axvline(
    estatistica_observada, color='#A23B3B', linewidth=2,
    label=f'Estatística observada = {estatistica_observada:.2f}',
)
ax.axvline(-abs(estatistica_observada), color='#A23B3B', linestyle='--', linewidth=1.5)
ax.axvline(abs(estatistica_observada), color='#A23B3B', linestyle='--', linewidth=1.5)
ax.set(
    title='Distribuição nula empírica: diferença de médias',
    xlabel='Média(casein) - média(meatmeal)',
    ylabel='Densidade',
)
ax.legend()
plt.tight_layout()


**Leitura da distribuição nula.** O histograma mostra as diferenças de médias produzidas pela redistribuição dos rótulos, mantendo os pesos observados e os tamanhos dos grupos. As linhas tracejadas delimitam as duas caudas consideradas extremas no teste bilateral. O valor-p é a proporção corrigida de atribuições com diferença pelo menos tão extrema quanto a observada; ele não é a probabilidade de $$H_0$$ ser verdadeira.


## 5. Decisão e interpretação

A decisão compara o valor-p com o nível de significância definido antes de examinar os resultados.


In [ ]:
if p_permutacao <= ALPHA:
    decisao = 'Rejeitamos H0: há evidência contra a igualdade das distribuições.'
else:
    decisao = 'Não rejeitamos H0: não há evidência suficiente contra a igualdade das distribuições.'

print(f'α = {ALPHA:.2f}')
print(decisao)


**Interpretação.** Se o valor-p for maior que $$0{,}05,$$ não rejeitamos $$H_0$$. Isso não demonstra que as rações produzem a mesma distribuição de pesos; indica que, sob a hipótese nula e o esquema de intercambiabilidade adotado, uma diferença pelo menos tão extrema não é suficientemente rara ao nível de 5%. A conclusão também não estabelece causalidade sem informações sobre como os frangos foram alocados às rações.


## 6. Comparação com o teste t de Welch

O teste t de Welch tem outro alvo formal: igualdade de médias, sem exigir variâncias populacionais iguais. Ele usa uma distribuição de referência $$t$$ aproximada, enquanto a permutação constrói a referência por redistribuição dos rótulos. Comparar as duas análises é instrutivo, mas resultados semelhantes não dispensam examinar se o desenho e a hipótese justificam cada procedimento.


In [ ]:
t_welch, p_welch = ttest_ind(casein, meatmeal, equal_var=False)

print(f'Estatística t de Welch: {t_welch:.3f}')
print(f'p-valor bilateral de Welch: {p_welch:.4f}')
print(f'p-valor bilateral por permutação: {p_permutacao:.4f}')


## Conclusão

Neste estudo de caso, o procedimento foi:

1. formular uma hipótese compatível com a intercambiabilidade dos rótulos;
2. escolher a diferença de médias e a alternativa bilateral antes de permutar;
3. preservar os tamanhos dos grupos em cada redistribuição;
4. gerar a distribuição nula empírica e calcular o valor-p com correção para Monte Carlo;
5. interpretar a decisão no contexto e distinguir o teste de permutação do teste t de Welch.

### Exercício de extensão

Altere a estatística para a diferença de medianas. O que muda na distribuição nula e no valor-p? A nova escolha responde exatamente à mesma pergunta? Justifique.
